## Observations

observation construct when there are $x$ number of adversaries, $y$ number of good_agents and $z$ number of landmarks:\
adversary $= 4+2z+2(x-1)+4y$\
good_agent $= 4+2z+2x+4(y-1)$

### 1. Environment Setup and Imports

In [16]:
# Import necessary libraries
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from pettingzoo.mpe import simple_tag_v2
from collections import deque
from torch.utils.data import DataLoader, Dataset

### 2. Modules for simple_tag_v2

##### 2.1 Observation wrapper

In [17]:
# Define experiment parameters
NUM_ADVERSARIES = 4  # Total number of adversaries
NUM_GOOD_AGENTS = 1  # Number of good agents
NUM_LANDMARKS = 2    # Number of landmarks

CLASS_A_COUNT = 2    # Number of adversaries in Class A
CLASS_B_COUNT = NUM_ADVERSARIES - CLASS_A_COUNT  # Number of adversaries in Class B

COMM_RANGE_CLASS_A = 1.0  # Communication range for Class A adversaries
COMM_RANGE_CLASS_B = 1.5  # Communication range for Class B adversaries

EPISODES = 1000      # Number of training episodes
MAX_STEPS = 1000     # Max steps per episode


##### 2.2 Communication Mechanism  based on Eucledian distance

In [18]:
# Initialize the environment
env = simple_tag_v2.env(num_good=NUM_GOOD_AGENTS, num_adversaries=NUM_ADVERSARIES, num_obstacles=NUM_LANDMARKS)
env.reset()


##### 2.3Construction of Graph for Commuincation

In [19]:
# Observation Handler for Adversaries
def process_observations(observations):
    """
    Process observations for adversaries based on their class.
    Returns a dictionary with processed observations for each adversary.
    """
    processed_obs = {}
    for agent_id, obs in observations.items():
        if 'adversary' in agent_id:
            # Determine the class of the adversary
            agent_index = int(agent_id.split('_')[-1])
            if agent_index < CLASS_A_COUNT:
                agent_class = 'A'
                comm_range = COMM_RANGE_CLASS_A
            else:
                agent_class = 'B'
                comm_range = COMM_RANGE_CLASS_B

            # Extract self position and velocity
            self_vel = obs[:2]
            self_pos = obs[2:4]

            # Initialize processed observation
            processed = {'self_pos': self_pos, 'self_vel': self_vel, 'class_id': agent_class}

            if agent_class == 'A':
                # Class A: Include absolute positions of landmarks
                landmark_positions = []
                for i in range(NUM_LANDMARKS):
                    idx = 4 + 2 * i
                    landmark_rel_pos = obs[idx: idx + 2]
                    landmark_abs_pos = self_pos + landmark_rel_pos
                    landmark_positions.append(landmark_abs_pos)
                processed['landmark_positions'] = np.concatenate(landmark_positions)
            else:
                # Class B: Include absolute position and velocity of good agents
                idx = 4 + 2 * NUM_LANDMARKS
                good_agent_rel_pos = obs[idx: idx + 2]
                good_agent_abs_pos = self_pos + good_agent_rel_pos
                good_agent_vel = obs[idx + 2: idx + 4]
                processed['good_agent_pos'] = good_agent_abs_pos
                processed['good_agent_vel'] = good_agent_vel

            # One-hot class ID
            class_id_vector = np.zeros(2)
            if agent_class == 'A':
                class_id_vector[0] = 1
            else:
                class_id_vector[1] = 1
            processed['class_id_vector'] = class_id_vector

            processed_obs[agent_id] = processed
    return processed_obs


##### 2.4 Init simple_tag_v2 and wrapper

In [20]:
# Define Temporal Transformer Module
import math

class TemporalTransformer(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers):
        super(TemporalTransformer, self).__init__()
        self.pos_encoder = PositionalEncoding(model_dim)
        encoder_layers = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.input_linear = nn.Linear(input_dim, model_dim)
        self.model_dim = model_dim

    def forward(self, src):
        # src shape: [sequence_length, batch_size, input_dim]
        src = self.input_linear(src) * math.sqrt(self.model_dim)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src)
        return output

class PositionalEncoding(nn.Module):
    def __init__(self, model_dim, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, model_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, model_dim, 2).float() * (-math.log(10000.0) / model_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        if model_dim % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [sequence_length, batch_size, model_dim]
        x = x + self.pe[:x.size(0), :]
        return x


### 3.Agent Networks

##### 3.1 Actor Network

In [21]:
# Define Class-specific Actor Networks
class ClassActor(nn.Module):
    def __init__(self, input_dim, hidden_dim, action_dim):
        super(ClassActor, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, action_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        action_logits = self.fc2(x)
        return action_logits

# Instantiate actors for each class
action_dim = env.action_space('adversary_0').n
input_dim_A = 2 + 2 * NUM_LANDMARKS + 2  # self_pos, landmark_positions, class_id_vector
input_dim_B = 2 + 4 + 2  # self_pos, good_agent_pos & vel, class_id_vector

actor_class_A = ClassActor(input_dim=input_dim_A, hidden_dim=128, action_dim=action_dim)
actor_class_B = ClassActor(input_dim=input_dim_B, hidden_dim=128, action_dim=action_dim)


##### 3.2. Critic Network

In [22]:
# Function to gather neighbor information and process with TemporalTransformer
def prepare_actor_input(processed_obs, agent_id, temporal_transformer):
    """
    Prepare the input for the actor network of a given adversary.
    """
    agent_obs = processed_obs[agent_id]
    agent_class = agent_obs['class_id']
    class_id_vector = agent_obs['class_id_vector']
    self_pos = agent_obs['self_pos']
    
    # Determine communication range based on class
    comm_range = COMM_RANGE_CLASS_A if agent_class == 'A' else COMM_RANGE_CLASS_B
    
    # Gather neighbor information within communication range
    neighbor_infos = []
    for other_id, other_obs in processed_obs.items():
        if other_id != agent_id and 'adversary' in other_id:
            distance = np.linalg.norm(self_pos - other_obs['self_pos'])
            if distance <= comm_range:
                neighbor_info = np.concatenate([
                    other_obs['self_pos'],
                    other_obs['self_vel'],
                    other_obs['class_id_vector']
                ])
                neighbor_infos.append(neighbor_info)
    
    # Process neighbor information with TemporalTransformer
    if neighbor_infos:
        neighbor_tensor = torch.FloatTensor(neighbor_infos)  # Shape: [num_neighbors, feature_dim]
        neighbor_tensor = neighbor_tensor.unsqueeze(1)       # Shape: [seq_len, batch_size, feature_dim]
        transformer_output = temporal_transformer(neighbor_tensor)
        # Take the output of the last layer
        transformer_output = transformer_output[-1, 0, :]    # Shape: [model_dim]
    else:
        # No neighbors within communication range
        transformer_output = torch.zeros(temporal_transformer.model_dim)
    
    # Prepare actor input
    if agent_class == 'A':
        # For Class A adversaries
        landmark_positions = agent_obs['landmark_positions']
        actor_input = np.concatenate([self_pos, landmark_positions, class_id_vector])
    else:
        # For Class B adversaries
        good_agent_pos = agent_obs['good_agent_pos']
        good_agent_vel = agent_obs['good_agent_vel']
        actor_input = np.concatenate([self_pos, good_agent_pos, good_agent_vel, class_id_vector])
    
    actor_input = torch.FloatTensor(actor_input)
    # Combine actor input with transformer output
    actor_input = torch.cat([actor_input, transformer_output], dim=0)
    return actor_input


##### 3.3 Temporal Transformer

In [23]:
# Define Critic Networks
class CentralCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(CentralCritic, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)  # Outputs a scalar value
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        value = self.fc2(x)
        return value

class ClassCritic(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(ClassCritic, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        value = self.fc2(x)
        return value

# Instantiate critics
hidden_dim = 128

# Input dimensions for critics
input_dim_central = (2 + 2) * NUM_ADVERSARIES + (2 + 2) * NUM_GOOD_AGENTS + 2 * NUM_LANDMARKS
input_dim_class_A = (2 + 2) * CLASS_A_COUNT + 2 * NUM_LANDMARKS
input_dim_class_B = (2 + 2) * CLASS_B_COUNT + (2 + 2)

central_critic = CentralCritic(input_dim=input_dim_central, hidden_dim=hidden_dim)
critic_class_A = ClassCritic(input_dim=input_dim_class_A, hidden_dim=hidden_dim)
critic_class_B = ClassCritic(input_dim=input_dim_class_B, hidden_dim=hidden_dim)

# Option to toggle class-based critic value decomposition
USE_CLASS_BASED_CRITIC = True


In [24]:
# Global Information Aggregator for Critic
def get_global_info(processed_obs):
    """
    Aggregate global information from all adversaries and good agents for the critic.
    """
    global_info = []
    # Collect information from adversaries
    for agent_id, agent_obs in processed_obs.items():
        if 'adversary' in agent_id:
            info = np.concatenate([
                agent_obs['self_pos'],
                agent_obs['self_vel']
            ])
            global_info.append(info)
    # Collect information from good agents
    for agent in env.agents:
        if 'agent' in agent:
            obs = env.observe(agent)
            self_vel = obs[:2]
            self_pos = obs[2:4]
            info = np.concatenate([self_pos, self_vel])
            global_info.append(info)
    # Collect landmark positions (assuming known and fixed)
    landmark_positions = []
    for i in range(NUM_LANDMARKS):
        # Since landmarks are static, we can retrieve their positions from the environment
        landmark_pos = env.world.landmarks[i].state.p_pos
        landmark_positions.append(landmark_pos)
    global_info.append(np.concatenate(landmark_positions))
    
    global_info = np.concatenate(global_info)
    return torch.FloatTensor(global_info)


In [25]:
# Define Optimizers and Loss Functions
actor_optimizer_A = optim.Adam(actor_class_A.parameters(), lr=1e-3)
actor_optimizer_B = optim.Adam(actor_class_B.parameters(), lr=1e-3)

if USE_CLASS_BASED_CRITIC:
    critic_optimizer_A = optim.Adam(critic_class_A.parameters(), lr=1e-3)
    critic_optimizer_B = optim.Adam(critic_class_B.parameters(), lr=1e-3)
else:
    critic_optimizer = optim.Adam(central_critic.parameters(), lr=1e-3)

mse_loss = nn.MSELoss()


In [26]:
# Instantiate Temporal Transformer
input_dim_transformer = 2 + 2 + 2  # self_pos, self_vel, class_id_vector
model_dim = 64
num_heads = 4
num_layers = 2

temporal_transformer = TemporalTransformer(
    input_dim=input_dim_transformer,
    model_dim=model_dim,
    num_heads=num_heads,
    num_layers=num_layers
)


In [27]:
# Define Replay Buffer Class
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.position] = (
            state,
            action,
            reward,
            next_state,
            done
        )
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        batch = zip(*random.sample(self.buffer, batch_size))
        return [np.array(items) for items in batch]

    def __len__(self):
        return len(self.buffer)


In [28]:
# Initialize Replay Buffers for Each Class
BUFFER_CAPACITY = 10000
replay_buffer_A = ReplayBuffer(capacity=BUFFER_CAPACITY)
replay_buffer_B = ReplayBuffer(capacity=BUFFER_CAPACITY)


In [29]:
# Modify Training Loop to Include Training Step
import random

for episode in range(EPISODES):
    env.reset()
    total_reward = 0
    step = 0
    while True:
        observations = {agent: env.observe(agent) for agent in env.agents}
        processed_obs = process_observations(observations)
        actions = {}
        for agent_id in env.agents:
            if 'adversary' in agent_id:
                # Prepare actor input
                actor_input = prepare_actor_input(processed_obs, agent_id, temporal_transformer)
                agent_obs = processed_obs[agent_id]
                agent_class = agent_obs['class_id']
                if agent_class == 'A':
                    action_logits = actor_class_A(actor_input)
                else:
                    action_logits = actor_class_B(actor_input)
                action_prob = nn.Softmax(dim=-1)(action_logits)
                action = torch.multinomial(action_prob, num_samples=1).item()
                actions[agent_id] = action
            else:
                # Good agents move away from the closest adversary
                actions[agent_id] = env.action_space(agent_id).sample()
        # Step the environment
        env.step(actions)
        # Collect rewards and next observations
        rewards = {agent: env.rewards[agent] for agent in env.agents}
        dones = {agent: env.dones[agent] for agent in env.agents}
        next_observations = {agent: env.observe(agent) for agent in env.agents}
        next_processed_obs = process_observations(next_observations)
        
        # Store experiences
        for agent_id in env.agents:
            if 'adversary' in agent_id:
                agent_obs = processed_obs[agent_id]
                agent_class = agent_obs['class_id']
                if agent_class == 'A':
                    replay_buffer_A.push(
                        agent_obs, actions[agent_id], rewards[agent_id],
                        next_processed_obs[agent_id], dones[agent_id]
                    )
                else:
                    replay_buffer_B.push(
                        agent_obs, actions[agent_id], rewards[agent_id],
                        next_processed_obs[agent_id], dones[agent_id]
                    )
        
        total_reward += sum(rewards.values())
        step += 1
        if all(dones.values()) or step >= MAX_STEPS:
            break
    
    print(f"Episode {episode}, Total Reward: {total_reward}")
    
    # Training
    if len(replay_buffer_A) >= batch_size and len(replay_buffer_B) >= batch_size:
        # Sample a batch for each class
        batch_A = replay_buffer_A.sample(batch_size)
        batch_B = replay_buffer_B.sample(batch_size)
        # Training code to be added here
        pass


/tmp/ipykernel_121618/2463409273.py:29: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1678411187366/work/torch/csrc/utils/tensor_new.cpp:245.)
  neighbor_tensor = torch.FloatTensor(neighbor_infos)  # Shape: [num_neighbors, feature_dim]


RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x72 and 8x128)